# Test Trained WLASL Model on OSL Dataset

This notebook tests if your trained WLASL-100 model can recognize signs from the OSL (Omani Sign Language?) dataset.

**What this does:**
- Loads your trained WLASL-100 model checkpoint
- Processes videos from OSL-Words dataset
- Extracts pose keypoints using MediaPipe
- Runs inference and shows top-5 predictions for each video

**Note:** Since the model was trained on WLASL (American Sign Language), predictions on OSL may not be accurate.
This test is to verify if the data format is compatible and if the model can process the videos.

In [1]:
# 1. Setup
import os
import sys
from pathlib import Path

# Set working directory
uni_sign_dir = Path(r'c:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main')
os.chdir(uni_sign_dir)
if str(uni_sign_dir) not in sys.path:
    sys.path.insert(0, str(uni_sign_dir))
print(f"Working directory: {os.getcwd()}")

# Suppress TensorFlow warnings (MediaPipe uses TFLite)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Working directory: c:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main


In [2]:
# 2. Check dependencies
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

import cv2
print(f"OpenCV: {cv2.__version__}")

import mediapipe as mp
print(f"MediaPipe: {mp.__version__}")

import numpy as np
print("All dependencies OK!")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# 3. Configuration

# Path to your trained model checkpoint
CHECKPOINT_PATH = "out/wlasl100_finetuning/best_checkpoint.pth"

# Alternative: use stage2 pretrained if fine-tuned isn't available
# CHECKPOINT_PATH = "out/stage2_pretraining/best_checkpoint.pth"

# OSL Dataset paths
OSL_WORDS_PATH = Path(r"C:\Users\MOBPC\Downloads\FYP\Dataset\dataset\OSL-Words\rgb_format")
OSL_SENTENCES_PATH = Path(r"C:\Users\MOBPC\Downloads\FYP\Dataset\dataset\OSL-Sentences\rgb_format")

# Verify paths exist
print(f"Checkpoint exists: {Path(CHECKPOINT_PATH).exists()}")
print(f"OSL-Words exists: {OSL_WORDS_PATH.exists()}")
print(f"OSL-Sentences exists: {OSL_SENTENCES_PATH.exists()}")

# List videos
osl_words_videos = list(OSL_WORDS_PATH.glob("*.mp4"))
print(f"\nOSL-Words videos found: {len(osl_words_videos)}")
if osl_words_videos:
    print("Sample videos:")
    for v in osl_words_videos[:5]:
        print(f"  - {v.name}")

In [ ]:
# 4. WLASL-100 Class Labels
WLASL100_LABELS = [
    "book", "drink", "computer", "before", "chair", "go", "clothes", "who", 
    "candy", "cousin", "deaf", "fine", "help", "no", "thin", "walk", "year",
    "yes", "all", "black", "cool", "finish", "hot", "like", "many", "mother",
    "now", "orange", "school", "study", "thanksgiving", "what", "woman", "bed",
    "blue", "bowling", "can", "dog", "family", "fish", "graduate", "hat",
    "hearing", "kiss", "language", "later", "man", "meet", "need", "nice",
    "nurse", "pizza", "play", "right", "same", "shirt", "sorry", "stay",
    "table", "tell", "want", "white", "work", "write", "accident", "apple",
    "bird", "change", "color", "corn", "cow", "dance", "dark", "doctor",
    "eat", "enjoy", "forget", "give", "happy", "have", "hearing", "hospital",
    "hurt", "know", "learn", "lost", "medicine", "movie", "paint", "paper",
    "pink", "pull", "read", "red", "restaurant", "see", "sick", "sign",
    "student", "teacher", "time", "wrong"
]
print(f"Number of WLASL-100 classes: {len(WLASL100_LABELS)}")

In [ ]:
# 5. Pose Extractor using MediaPipe
import mediapipe as mediapipe_module

class PoseExtractor:
    """Extract pose keypoints using MediaPipe, matching the training data format"""

    # MediaPipe Pose landmark indices -> 9 body joints matching training format
    BODY_MP_INDICES = [0, 7, 8, 11, 12, 13, 14, 15, 16]

    # MediaPipe Face Mesh indices -> 18 face keypoints
    FACE_JAW_MP = [234, 93, 132, 58, 172, 136, 150, 176, 152]  # 9 pts
    FACE_MOUTH_MP = [78, 191, 80, 81, 82, 13, 312, 311]         # 8 pts
    FACE_NOSE_MP = [1]                                          # 1 pt

    def __init__(self):
        self.mp_holistic = mediapipe_module.solutions.holistic
        self.holistic = self.mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
            model_complexity=1
        )
        self.face_indices = self.FACE_JAW_MP + self.FACE_MOUTH_MP + self.FACE_NOSE_MP
        print("[OK] MediaPipe Holistic initialized")

    def extract(self, frame):
        """Extract pose keypoints from a single frame"""
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.holistic.process(rgb_frame)

        # Body: 9 joints
        body = np.zeros((9, 3), dtype=np.float32)
        if results.pose_landmarks:
            for i, mp_idx in enumerate(self.BODY_MP_INDICES):
                lm = results.pose_landmarks.landmark[mp_idx]
                body[i] = [lm.x, lm.y, lm.visibility]

        # Left hand: 21 joints
        left_hand = np.zeros((21, 3), dtype=np.float32)
        if results.left_hand_landmarks:
            for i, lm in enumerate(results.left_hand_landmarks.landmark):
                left_hand[i] = [lm.x, lm.y, 1.0]
            left_hand[:, :2] -= left_hand[0, :2].copy()

        # Right hand: 21 joints
        right_hand = np.zeros((21, 3), dtype=np.float32)
        if results.right_hand_landmarks:
            for i, lm in enumerate(results.right_hand_landmarks.landmark):
                right_hand[i] = [lm.x, lm.y, 1.0]
            right_hand[:, :2] -= right_hand[0, :2].copy()

        # Face: 18 joints
        face = np.zeros((18, 3), dtype=np.float32)
        if results.face_landmarks:
            for i, mp_idx in enumerate(self.face_indices):
                if mp_idx < len(results.face_landmarks.landmark):
                    lm = results.face_landmarks.landmark[mp_idx]
                    face[i] = [lm.x, lm.y, 1.0]
            face[:, :2] -= face[-1, :2].copy()

        return {
            'body': body,
            'left': left_hand,
            'right': right_hand,
            'face_all': face
        }
    
    def close(self):
        self.holistic.close()

print("PoseExtractor class defined")

In [ ]:
# 6. Load the trained model
from models import Uni_Sign
import argparse

class SignRecognizer:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")
        
        # Model configuration
        self.args = argparse.Namespace(
            hidden_dim=256,
            rgb_support=False,  # Pose-only for inference
            dataset='WLASL',
            task='ISLR',
            max_length=64,
            label_smoothing=0.0
        )
        
        print("Loading Uni-Sign model...")
        self.model = Uni_Sign(args=self.args)
        
        # Load checkpoint
        if Path(checkpoint_path).exists():
            print(f"Loading checkpoint: {checkpoint_path}")
            state_dict = torch.load(checkpoint_path, map_location='cpu')['model']
            model_dict = self.model.state_dict()
            filtered_dict = {k: v for k, v in state_dict.items() if k in model_dict}
            self.model.load_state_dict(filtered_dict, strict=False)
            print(f"[OK] Loaded {len(filtered_dict)}/{len(state_dict)} weights")
        else:
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
        
        self.model.to(self.device)
        self.model.eval()
        self.labels = WLASL100_LABELS
        print(f"[OK] Model ready with {len(self.labels)} classes")
    
    def predict_from_poses(self, pose_frames):
        """Run prediction on a list of pose frames"""
        if len(pose_frames) < 4:
            return None, 0.0, "Not enough frames"
        
        T = len(pose_frames)
        thr = 0.3
        
        # Stack poses
        body_all = np.stack([f['body'] for f in pose_frames])
        left_all = np.stack([f['left'] for f in pose_frames])
        right_all = np.stack([f['right'] for f in pose_frames])
        face_all = np.stack([f['face_all'] for f in pose_frames])
        
        # Normalize body poses
        body_xy = body_all[:, :, :2].reshape(-1, 2)
        body_conf = body_all[:, :, 2].reshape(-1)
        valid = body_conf > thr
        if valid.sum() > 0:
            xy_valid = body_xy[valid]
            min_xy = xy_valid.min(axis=0)
            max_xy = xy_valid.max(axis=0)
            center = (min_xy + max_xy) / 2
            scale = max(max_xy - min_xy) + 1e-6
            body_all[:, :, :2] = (body_all[:, :, :2] - center) / scale
        
        # Prepare batch
        batch = {
            'body': torch.tensor(body_all, dtype=torch.float32).unsqueeze(0),
            'left': torch.tensor(left_all, dtype=torch.float32).unsqueeze(0),
            'right': torch.tensor(right_all, dtype=torch.float32).unsqueeze(0),
            'face_all': torch.tensor(face_all, dtype=torch.float32).unsqueeze(0),
        }
        
        # Move to device
        for k in batch:
            batch[k] = batch[k].to(self.device)
        
        # Run inference
        with torch.no_grad():
            try:
                # Encode pose features
                visual_feat = self.model.encode(batch)
                
                # Get logits from classifier head
                logits = self.model.classifier_head(visual_feat)
                probs = torch.softmax(logits, dim=-1)
                
                # Get top-5 predictions
                top5_probs, top5_indices = torch.topk(probs[0], k=min(5, len(self.labels)))
                
                results = []
                for prob, idx in zip(top5_probs.cpu().numpy(), top5_indices.cpu().numpy()):
                    results.append((self.labels[idx], float(prob)))
                
                return results
            except Exception as e:
                return None, 0.0, str(e)

print("SignRecognizer class defined")

In [ ]:
# 7. Initialize the model and pose extractor
print("Initializing...")

pose_extractor = PoseExtractor()
recognizer = SignRecognizer(CHECKPOINT_PATH)

print("\n[OK] Ready to process videos!")

In [ ]:
# 8. Process a single video
def process_video(video_path, pose_extractor, max_frames=64, sample_rate=2):
    """Extract poses from a video file"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Error: Cannot open {video_path}")
        return None
    
    poses = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Sample frames
        if frame_idx % sample_rate == 0:
            pose = pose_extractor.extract(frame)
            poses.append(pose)
            
            if len(poses) >= max_frames:
                break
        
        frame_idx += 1
    
    cap.release()
    return poses

# Test with first video
if osl_words_videos:
    test_video = osl_words_videos[0]
    print(f"Testing with: {test_video.name}")
    
    poses = process_video(test_video, pose_extractor)
    if poses:
        print(f"Extracted {len(poses)} frames")
        
        # Run prediction
        results = recognizer.predict_from_poses(poses)
        if results:
            print(f"\nTop-5 Predictions for {test_video.name}:")
            for i, (label, conf) in enumerate(results, 1):
                print(f"  {i}. {label}: {conf*100:.1f}%")

In [ ]:
# 9. Process multiple videos from OSL-Words
print("Processing OSL-Words videos...")
print("=" * 70)

# Process first N videos
NUM_VIDEOS_TO_TEST = 10  # Change this to test more videos

results_df = []
for i, video_path in enumerate(osl_words_videos[:NUM_VIDEOS_TO_TEST]):
    print(f"\n[{i+1}/{NUM_VIDEOS_TO_TEST}] {video_path.name}")
    
    poses = process_video(video_path, pose_extractor, max_frames=64, sample_rate=2)
    
    if poses and len(poses) >= 4:
        results = recognizer.predict_from_poses(poses)
        if results:
            top_pred, top_conf = results[0]
            print(f"  Frames: {len(poses)} | Top prediction: {top_pred} ({top_conf*100:.1f}%)")
            print(f"  Top-3: {', '.join([f'{l}({c*100:.0f}%)' for l,c in results[:3]])}")
            
            results_df.append({
                'video': video_path.name,
                'frames': len(poses),
                'prediction_1': results[0][0],
                'confidence_1': results[0][1],
                'prediction_2': results[1][0] if len(results) > 1 else '',
                'prediction_3': results[2][0] if len(results) > 2 else ''
            })
        else:
            print(f"  Prediction failed")
    else:
        print(f"  Not enough frames extracted")

print("\n" + "=" * 70)
print(f"Processed {len(results_df)} videos successfully")

In [ ]:
# 10. Summary of predictions
import pandas as pd

if results_df:
    df = pd.DataFrame(results_df)
    print("\nPrediction Summary:")
    print("=" * 70)
    display(df)
    
    # Statistics
    print(f"\nStatistics:")
    print(f"  Average confidence: {df['confidence_1'].mean()*100:.1f}%")
    print(f"  Max confidence: {df['confidence_1'].max()*100:.1f}%")
    print(f"  Min confidence: {df['confidence_1'].min()*100:.1f}%")
    
    # Most common predictions
    print(f"\nMost common predictions:")
    pred_counts = df['prediction_1'].value_counts().head(10)
    for pred, count in pred_counts.items():
        print(f"  {pred}: {count} videos")
else:
    print("No results to display")

## Interpretation

**If you see predictions with reasonable confidence (>50%):**
- The model successfully processes the OSL video format
- Pose extraction is working correctly
- You could potentially fine-tune/retrain on OSL data

**If predictions have low confidence or seem random:**
- This is expected since ASL and OSL are different sign languages
- The models architecture works, but needs training on OSL data
- You would need to create OSL labels and fine-tune the model

**Next steps to use this model with OSL:**
1. Create label files mapping OSL video names to sign classes
2. Extract poses for OSL videos (or use RGB directly)
3. Fine-tune the model on OSL data using similar process as WLASL

In [ ]:
# 11. Optional: Test on OSL-Sentences
osl_sentences_videos = list(OSL_SENTENCES_PATH.glob("*.mp4"))
print(f"OSL-Sentences videos found: {len(osl_sentences_videos)}")

if osl_sentences_videos and len(osl_sentences_videos) > 0:
    print("\nTesting one sentence video:")
    test_video = osl_sentences_videos[0]
    print(f"Video: {test_video.name}")
    
    poses = process_video(test_video, pose_extractor, max_frames=128, sample_rate=3)
    if poses:
        print(f"Extracted {len(poses)} frames")
        results = recognizer.predict_from_poses(poses)
        if results:
            print(f"\nNote: ISLR model predicts single signs, not sentences.")
            print(f"Top-5 predictions:")
            for i, (label, conf) in enumerate(results, 1):
                print(f"  {i}. {label}: {conf*100:.1f}%")

In [ ]:
# 12. Cleanup
pose_extractor.close()
print("Done! MediaPipe resources released.")